# Notebook 03 — Single-Frame Detection and Visualisation

## What this notebook does
I use this notebook to run YOLOv8 detection on individual video frames, inspect the raw detection outputs (bounding boxes, class IDs, confidence scores), and produce annotated visualisations. This is the foundational step before building the full video pipeline.

## Why this matters
Frame-level inspection lets me:
1. Confirm the model is detecting objects correctly before processing a full video.
2. Understand the detection output format (bounding box coordinates, class IDs, confidence).
3. Tune the confidence threshold by observing which detections look correct vs. spurious.
4. Visualise the bounding-box drawing logic before applying it to video.

## Input files expected
- `models/yolov8s.pt` (from Notebook 01)
- `data/raw/sample_traffic.mp4` OR `data/mock/mock_video.mp4`

## Output files created
- `reports/figures/03_single_frame_detections.png`
- `reports/figures/03_confidence_distribution.png`
- `data/interim/frame_0050.jpg` — sample extracted frame

## Connection to the research question
Understanding single-frame detection output is the prerequisite for the counting logic and the video pipeline.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2

from detection_system.loader import load_model
from detection_system.inference import detect_frame
from detection_system.visualize import draw_detections, bgr_to_rgb, resize_frame
from detection_system.utils import load_config, save_figure

cfg = load_config("config.yaml")
plt.style.use(cfg["figures"]["style"])
CONF_THRESHOLD = cfg["model"]["confidence_threshold"]
print(f"Confidence threshold: {CONF_THRESHOLD}")
print("Setup complete")

Confidence threshold: 0.25
Setup complete


In [2]:
# ── Step 1: Load model and select video ───────────────────────────────────────
model = load_model("small", models_dir=PROJECT_ROOT / "models")

# Choose real video if available, otherwise fall back to mock
real_video = PROJECT_ROOT / "data" / "raw" / "sample_traffic.mp4"
mock_video = PROJECT_ROOT / "data" / "mock" / "mock_video.mp4"

if real_video.exists():
    VIDEO_PATH = real_video
    print(f"Using real video: {VIDEO_PATH.name}")
else:
    VIDEO_PATH = mock_video
    print(f"Using mock video: {VIDEO_PATH.name}")
    print("NOTE: Mock video — detection results will be minimal (no real objects).")

Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 SMALL | Classes: 80
Using real video: sample_traffic.mp4


In [3]:
# ── Step 2: Extract sample frames from the video ──────────────────────────────
# I extract frames at three time points: start, middle, and end of the video.
# This gives a representative sample of the scene content.

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

cap = cv2.VideoCapture(str(VIDEO_PATH))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

# Sample at 10%, 50%, and 90% through the video
sample_positions = {
    "early":  int(total_frames * 0.10),
    "middle": int(total_frames * 0.50),
    "late":   int(total_frames * 0.90),
}

sample_frames = {}
for label, frame_idx in sample_positions.items():
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if ret:
        sample_frames[label] = frame
        # Save to interim/
        save_path = INTERIM_DIR / f"frame_{frame_idx:04d}.jpg"
        cv2.imwrite(str(save_path), frame)
        print(f"Extracted frame {frame_idx} ({label}) → {save_path.name}")

cap.release()
print(f"\nExtracted {len(sample_frames)} sample frames")

Extracted frame 150 (early) → frame_0150.jpg
Extracted frame 750 (middle) → frame_0750.jpg
Extracted frame 1350 (late) → frame_1350.jpg

Extracted 3 sample frames


In [4]:
# ── Step 3: Run detection on the middle frame ──────────────────────────────────
# I focus on the middle frame for the initial inspection.

frame_to_analyse = sample_frames.get("middle", list(sample_frames.values())[0])
h, w = frame_to_analyse.shape[:2]
print(f"Frame shape: {w}×{h} pixels (W×H)")

detections, latency = detect_frame(model, frame_to_analyse, confidence=CONF_THRESHOLD)

print(f"\nDetection results:")
print(f"  Total detections : {len(detections)}")
print(f"  Inference time   : {latency:.1f} ms")
print(f"  Equiv. FPS       : {1000/latency:.1f}")

Frame shape: 1920×1080 pixels (W×H)

Detection results:
  Total detections : 8
  Inference time   : 407.4 ms
  Equiv. FPS       : 2.5


In [5]:
# ── Step 4: Inspect raw detection output ──────────────────────────────────────
# I print each detection as a structured dict to understand the format
# before building the counting and visualisation logic.

print(f"Individual detections (confidence > {CONF_THRESHOLD}):")
print(f"{'#':<4} {'Class':<16} {'Conf':>6}  {'BBox (x1,y1,x2,y2)'}")
print("-" * 65)

for i, det in enumerate(detections):
    x1, y1, x2, y2 = [int(v) for v in det["bbox_xyxy"]]
    conf = det["confidence"]
    cls = det["class_name"]
    print(f"{i:<4} {cls:<16} {conf:>6.3f}  ({x1}, {y1}, {x2}, {y2})")

# Convert to DataFrame for easier inspection
if detections:
    det_df = pd.DataFrame(detections)
    # Expand bbox_xyxy into separate columns
    det_df[["x1", "y1", "x2", "y2"]] = pd.DataFrame(
        det_df["bbox_xyxy"].tolist(), index=det_df.index
    )
    det_df = det_df.drop(columns=["bbox_xyxy", "bbox_xywh"])
    print(f"\nDetection DataFrame shape: {det_df.shape}")
    display(det_df.head(10))

Individual detections (confidence > 0.25):
#    Class              Conf  BBox (x1,y1,x2,y2)
-----------------------------------------------------------------
0    car               0.910  (948, 314, 1171, 486)
1    car               0.903  (522, 317, 726, 490)
2    car               0.808  (446, 283, 614, 420)
3    car               0.784  (294, 274, 424, 363)
4    car               0.781  (776, 263, 945, 405)
5    truck             0.475  (135, 204, 261, 359)
6    truck             0.379  (609, 263, 725, 353)
7    car               0.367  (607, 264, 725, 351)

Detection DataFrame shape: (8, 7)


,class_id,class_name,confidence,x1,y1,x2,y2
0,2,car,0.909890,948.562683,314.049042,1171.214355,486.323273
1,2,car,0.902521,522.924011,317.362579,726.308044,490.009644
2,2,car,0.807932,446.982056,283.568024,614.811829,420.181732
3,2,car,0.784477,294.435242,274.196503,424.604218,363.762085
4,2,car,0.780771,776.686340,263.295990,945.195007,405.110718
5,7,truck,0.475276,135.217438,204.599533,261.406677,359.830017
6,7,truck,0.378700,609.618958,263.771027,725.055908,353.692566
7,2,car,0.367209,607.346252,264.480194,725.651245,351.175323


In [6]:
# ── Step 5: Visualise detections on the frame ─────────────────────────────────
# I draw bounding boxes on the frame and display it inline.
# Note: cv2 uses BGR colour order; matplotlib expects RGB.
# I use bgr_to_rgb() before plt.imshow() to get correct colours.

annotated_frame = draw_detections(frame_to_analyse.copy(), detections)
annotated_rgb = bgr_to_rgb(annotated_frame)  # BGR → RGB for matplotlib

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original frame
axes[0].imshow(bgr_to_rgb(frame_to_analyse))
axes[0].set_title("Original Frame", fontsize=12)
axes[0].axis("off")

# Annotated frame
axes[1].imshow(annotated_rgb)
axes[1].set_title(f"YOLOv8-small Detections ({len(detections)} objects, conf≥{CONF_THRESHOLD})", fontsize=12)
axes[1].axis("off")

plt.suptitle("Single-Frame Object Detection — Middle Sample Frame", fontsize=14, fontweight="bold")
plt.tight_layout()

save_figure(fig, "03_single_frame_detections")
plt.show()

  Saved: reports\figures\03_single_frame_detections.png
  Saved: reports\figures\03_single_frame_detections.pdf
  Saved: paper\figures\03_single_frame_detections.png
  Saved: paper\figures\03_single_frame_detections.pdf


In [7]:
# ── Step 6: Detection confidence distribution ─────────────────────────────────
# I inspect the distribution of confidence scores to inform the threshold choice.
# A good threshold balances precision (fewer false positives) and recall
# (fewer missed objects).

# Run detection across all 3 sample frames to get more data
all_detections_multi = []
for label, frame in sample_frames.items():
    dets, _ = detect_frame(model, frame, confidence=0.05)  # Low threshold to see full distribution
    for d in dets:
        all_detections_multi.append({"frame": label, **d})

if all_detections_multi:
    conf_df = pd.DataFrame(all_detections_multi)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Confidence histogram
    axes[0].hist(conf_df["confidence"], bins=30, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].axvline(CONF_THRESHOLD, color="crimson", lw=2, ls="--",
                    label=f"Threshold = {CONF_THRESHOLD}")
    axes[0].set_xlabel("Confidence score", fontsize=12)
    axes[0].set_ylabel("Number of detections", fontsize=12)
    axes[0].set_title("Detection Confidence Distribution", fontsize=12)
    axes[0].legend()
    
    # Per-class confidence box plot (top 8 classes by count)
    top_classes = conf_df["class_name"].value_counts().head(8).index
    conf_filtered = conf_df[conf_df["class_name"].isin(top_classes)]
    
    sns.boxplot(
        data=conf_filtered,
        x="confidence",
        y="class_name",
        palette="colorblind",
        ax=axes[1],
        order=top_classes,
    )
    axes[1].axvline(CONF_THRESHOLD, color="crimson", lw=2, ls="--",
                    label=f"Threshold = {CONF_THRESHOLD}")
    axes[1].set_xlabel("Confidence score", fontsize=12)
    axes[1].set_ylabel("Class", fontsize=12)
    axes[1].set_title("Per-Class Confidence Distribution", fontsize=12)
    axes[1].legend()
    
    plt.suptitle("Confidence Score Analysis (conf ≥ 0.05, 3 sample frames)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    
    save_figure(fig, "03_confidence_distribution")
    plt.show()
    
    # Summary table
    print("\nDetection summary across 3 sample frames:")
    print(conf_df.groupby("class_name")["confidence"].describe().round(3))
else:
    print("No detections found at conf ≥ 0.05. Video may be blank/mock.")
    print("This is expected for mock video. Continue with Notebook 04.")

C:\Users\Peter\AppData\Local\Temp\ipykernel_16548\2793827583.py:31: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(


  Saved: reports\figures\03_confidence_distribution.png
  Saved: reports\figures\03_confidence_distribution.pdf
  Saved: paper\figures\03_confidence_distribution.png
  Saved: paper\figures\03_confidence_distribution.pdf

Detection summary across 3 sample frames:
            count   mean    std    min    25%    50%    75%    max
class_name                                                        
car          20.0  0.610  0.325  0.072  0.326  0.748  0.859  0.914
person        6.0  0.066  0.010  0.053  0.058  0.071  0.074  0.075
truck         9.0  0.238  0.188  0.052  0.105  0.118  0.379  0.563


In [8]:
# ── Step 7: Threshold sensitivity check ───────────────────────────────────────
# I check how many detections survive at different confidence thresholds.
# This informs the threshold choice: too low = many false positives,
# too high = many missed objects.

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]
threshold_results = []

for thresh in thresholds:
    count = 0
    for frame in sample_frames.values():
        dets, _ = detect_frame(model, frame, confidence=thresh)
        count += len(dets)
    threshold_results.append({"threshold": thresh, "total_detections": count})

thresh_df = pd.DataFrame(threshold_results)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresh_df["threshold"], thresh_df["total_detections"],
        marker="o", color="steelblue", linewidth=2)
ax.axvline(CONF_THRESHOLD, color="crimson", lw=2, ls="--",
           label=f"Chosen threshold = {CONF_THRESHOLD}")
ax.set_xlabel("Confidence threshold", fontsize=12)
ax.set_ylabel("Total detections (3 frames)", fontsize=12)
ax.set_title("Detection Count vs. Confidence Threshold", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()

save_figure(fig, "03_threshold_sensitivity")
plt.show()

print("\nThreshold sensitivity table:")
print(thresh_df.to_string(index=False))
print(f"\nChosen threshold: {CONF_THRESHOLD} (set in configs/config.yaml)")

  Saved: reports\figures\03_threshold_sensitivity.png
  Saved: reports\figures\03_threshold_sensitivity.pdf
  Saved: paper\figures\03_threshold_sensitivity.png
  Saved: paper\figures\03_threshold_sensitivity.pdf

Threshold sensitivity table:
 threshold  total_detections
      0.10                24
      0.15                20
      0.20                20
      0.25                18
      0.30                18
      0.40                16
      0.50                15
      0.60                14
      0.70                12

Chosen threshold: 0.25 (set in configs/config.yaml)
